In [2]:
from pynq import Overlay
ol = Overlay("/home/xilinx/jupyter_notebooks/video/camera_test.bit")
print(ol.ip_dict.keys())

dict_keys(['video/hdmi_in/frontend/axi_gpio_hdmiin', 'video/hdmi_out/frontend/hdmi_out_hpd_video', 'system_interrupts', 'video/axi_vdma', 'video/hdmi_in/color_convert', 'video/hdmi_in/frontend/vtc_in', 'video/hdmi_in/pixel_pack', 'video/hdmi_out/color_convert', 'video/hdmi_out/frontend/axi_dynclk', 'video/hdmi_out/frontend/vtc_out', 'video/hdmi_out/pixel_unpack', 'ps7_0'])


In [3]:
from pynq import Overlay
ol = Overlay("/home/xilinx/jupyter_notebooks/video/design_1.bit")
print(ol.ip_dict.keys())

dict_keys(['axi_vdma_0', 'v_tpg_0', 'ps7'])


In [9]:
from pynq import Overlay
ol = Overlay("/home/xilinx/jupyter_notebooks/video/cam.bit")
print(ol.ip_dict.keys())

dict_keys(['axi_vdma_0', 'v_tpg_0', 'ps7'])


In [10]:
from pynq import Overlay
ol = Overlay("/home/xilinx/jupyter_notebooks/video/camera_hdmi.bit")
print(ol.ip_dict.keys())

dict_keys(['axi_gpio_0', 'axi_vdma_0', 'v_tc_0', 'processing_system7_0'])


In [2]:
import cv2, time
cap = cv2.VideoCapture("/dev/video0", cv2.CAP_V4L2)
time.sleep(0.5)
print("opened:", cap.isOpened())
cap.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter_fourcc(*'MJPG'))
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
for i in range(15):
    ok, f = cap.read()
    print(i, ok, None if f is None else f.shape)
    time.sleep(0.05)
cap.release()

opened: True
0 True (720, 1280, 3)
1 True (720, 1280, 3)
2 True (720, 1280, 3)
3 True (720, 1280, 3)
4 True (720, 1280, 3)
5 True (720, 1280, 3)
6 True (720, 1280, 3)
7 True (720, 1280, 3)
8 True (720, 1280, 3)
9 True (720, 1280, 3)
10 True (720, 1280, 3)
11 True (720, 1280, 3)
12 True (720, 1280, 3)
13 True (720, 1280, 3)
14 True (720, 1280, 3)


In [1]:
import time
import numpy as np
from pynq import Overlay, allocate, MMIO

ol = Overlay("/home/xilinx/jupyter_notebooks/video/design_1.bit")
print(ol.ip_dict.keys())   # attendu: axi_vdma_0, v_tpg_0, ps7

# --- paramètres 720p, RGB 24 bits (3 octets/pixel) ---
WIDTH, HEIGHT, BPP = 1280, 720, 3
stride = WIDTH * BPP            # 3840 octets par ligne

# --- framebuffer contigu en DRAM ---
fb = allocate(shape=(HEIGHT, WIDTH, BPP), dtype=np.uint8)

# mire en barres de couleur
bars = [(255,255,255),(255,255,0),(0,255,255),(0,255,0),
        (255,0,255),(255,0,0),(0,0,255),(0,0,0)]
bw = WIDTH // len(bars)
for i,(r,g,b) in enumerate(bars):
    fb[:, i*bw:(i+1)*bw, 0] = r
    fb[:, i*bw:(i+1)*bw, 1] = g
    fb[:, i*bw:(i+1)*bw, 2] = b
fb.flush()

# --- registres VDMA (canal MM2S) ---
vdma_base = ol.ip_dict['axi_vdma_0']['phys_addr']
vdma = MMIO(vdma_base, 0x100)

MM2S_DMACR        = 0x00
MM2S_DMASR        = 0x04
MM2S_VSIZE        = 0x50   # écrire VSIZE déclenche le transfert
MM2S_HSIZE        = 0x54
MM2S_FRMDLY_STRIDE= 0x58
MM2S_START_ADDR   = 0x5C   # frame 0 (puis +4, +8 pour frames 1,2)

addr = fb.physical_address

# reset du canal MM2S
vdma.write(MM2S_DMACR, 0x4)
time.sleep(0.01)
# run, mode circulaire
vdma.write(MM2S_DMACR, 0x1)
# même adresse dans 3 slots -> affiche toujours notre frame
vdma.write(MM2S_START_ADDR + 0, addr)
vdma.write(MM2S_START_ADDR + 4, addr)
vdma.write(MM2S_START_ADDR + 8, addr)
# stride + hsize (en octets), puis vsize (lignes)
vdma.write(MM2S_FRMDLY_STRIDE, stride)
vdma.write(MM2S_HSIZE, WIDTH * BPP)
vdma.write(MM2S_VSIZE, HEIGHT)

time.sleep(0.1)
print("DMASR = 0x{:08X}".format(vdma.read(MM2S_DMASR)))

dict_keys(['axi_vdma_0', 'v_tpg_0', 'ps7'])
DMASR = 0x00011000


## Video without Offset

In [ ]:
import time, cv2
import numpy as np
from pynq import Overlay, allocate, MMIO

ol = Overlay("/home/xilinx/jupyter_notebooks/video/design_1.bit")

WIDTH, HEIGHT, BPP = 1280, 720, 3
stride = WIDTH * BPP

fb = allocate(shape=(HEIGHT, WIDTH, BPP), dtype=np.uint8)
fb[:] = 0; fb.flush()
addr = fb.physical_address

vdma = MMIO(ol.ip_dict['axi_vdma_0']['phys_addr'], 0x100)
MM2S_DMACR, MM2S_DMASR = 0x00, 0x04
MM2S_VSIZE, MM2S_HSIZE = 0x50, 0x54
MM2S_STRIDE, MM2S_ADDR = 0x58, 0x5C

def vdma_start():
    vdma.write(MM2S_DMACR, 0x4)   # reset
    time.sleep(0.2)               # laisser le VTC tourner seul
    vdma.write(MM2S_DMACR, 0x1)   # run
    for i in range(3):
        vdma.write(MM2S_ADDR + i*4, addr)
    vdma.write(MM2S_STRIDE, stride)
    vdma.write(MM2S_HSIZE, WIDTH * BPP)
    vdma.write(MM2S_VSIZE, HEIGHT)

# ouverture par chemin explicite + warm-up (la combinaison qui marche)
cap = cv2.VideoCapture("/dev/video0", cv2.CAP_V4L2)
time.sleep(0.5)
cap.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter_fourcc(*'MJPG'))
cap.set(cv2.CAP_PROP_FRAME_WIDTH, WIDTH)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, HEIGHT)
for _ in range(10):
    cap.read(); time.sleep(0.05)

vdma_start()

fail = 0
t0 = time.time(); n = 0
try:
    while True:
        ok, frame = cap.read()
        if not ok or frame is None:
            fail += 1
            if fail % 30 == 0:
                print("read fail x", fail)
            continue
        fail = 0
        if frame.shape[1] != WIDTH or frame.shape[0] != HEIGHT:
            frame = cv2.resize(frame, (WIDTH, HEIGHT))
        fb[:] = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        fb.flush()
        n += 1
        if n % 30 == 0:
            print("fps ~ {:.1f}".format(n / (time.time() - t0)))
except KeyboardInterrupt:
    pass
finally:
    cap.release()
    vdma.write(MM2S_DMACR, 0x4)
    print("stopped, DMASR = 0x{:08X}".format(vdma.read(MM2S_DMASR)))

fps ~ 9.2
fps ~ 9.5
fps ~ 9.6
fps ~ 9.6
fps ~ 9.6
fps ~ 9.6
fps ~ 9.6
fps ~ 9.6
fps ~ 9.6
fps ~ 9.6
fps ~ 9.6
fps ~ 9.6
fps ~ 9.6
fps ~ 9.6
fps ~ 9.6
fps ~ 9.6
fps ~ 9.5
fps ~ 9.5
fps ~ 9.5
fps ~ 9.5
fps ~ 9.5
fps ~ 9.5
fps ~ 9.5
fps ~ 9.5
fps ~ 9.5
fps ~ 9.5
fps ~ 9.5
fps ~ 9.5
fps ~ 9.5
fps ~ 9.5
fps ~ 9.5
fps ~ 9.6
fps ~ 9.6
fps ~ 9.6
fps ~ 9.6
fps ~ 9.6
fps ~ 9.6
fps ~ 9.6
fps ~ 9.6
fps ~ 9.6
fps ~ 9.6


## Video TEst with Offset = 299 px

In [ ]:
import time, cv2
import numpy as np
from pynq import Overlay, allocate, MMIO

ol = Overlay("/home/xilinx/jupyter_notebooks/video/design_1.bit")

WIDTH, HEIGHT, BPP = 1280, 720, 3
stride = WIDTH * BPP
OFF = 300 #256          # offset horizontal a compenser (ajuste apres mesure)

fb = allocate(shape=(HEIGHT, WIDTH, BPP), dtype=np.uint8)
fb[:] = 0; fb.flush()
addr = fb.physical_address

vdma = MMIO(ol.ip_dict['axi_vdma_0']['phys_addr'], 0x100)
MM2S_DMACR, MM2S_DMASR = 0x00, 0x04
MM2S_VSIZE, MM2S_HSIZE = 0x50, 0x54
MM2S_STRIDE, MM2S_ADDR = 0x58, 0x5C

def vdma_start():
    vdma.write(MM2S_DMACR, 0x4); time.sleep(0.2)
    vdma.write(MM2S_DMACR, 0x1)
    for i in range(3):
        vdma.write(MM2S_ADDR + i*4, addr)
    vdma.write(MM2S_STRIDE, stride)
    vdma.write(MM2S_HSIZE, WIDTH * BPP)
    vdma.write(MM2S_VSIZE, HEIGHT)

cap = cv2.VideoCapture("/dev/video0", cv2.CAP_V4L2)
time.sleep(0.5)
cap.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter_fourcc(*'MJPG'))
cap.set(cv2.CAP_PROP_FRAME_WIDTH, WIDTH)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, HEIGHT)
for _ in range(10):
    cap.read(); time.sleep(0.05)

vdma_start()

fail = 0
t0 = time.time(); n = 0
try:
    while True:
        ok, frame = cap.read()
        if not ok or frame is None:
            fail += 1
            if fail % 30 == 0:
                print("read fail x", fail)
            continue
        fail = 0
        if frame.shape[1] != WIDTH or frame.shape[0] != HEIGHT:
            frame = cv2.resize(frame, (WIDTH, HEIGHT))
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        fb[:] = np.roll(rgb, -OFF, axis=1)   # compense le decalage horizontal
        fb.flush()
        n += 1
        if n % 30 == 0:
            print("fps ~ {:.1f}".format(n / (time.time() - t0)))
except KeyboardInterrupt:
    pass
finally:
    cap.release()
    vdma.write(MM2S_DMACR, 0x4)
    print("stopped, DMASR = 0x{:08X}".format(vdma.read(MM2S_DMASR)))

fps ~ 7.7
fps ~ 7.7
fps ~ 7.8
fps ~ 7.9
fps ~ 7.8
fps ~ 7.8
fps ~ 7.8
fps ~ 7.9
fps ~ 7.9
fps ~ 7.9
fps ~ 7.9
fps ~ 7.9
fps ~ 7.9
fps ~ 7.9
fps ~ 7.9
fps ~ 7.9
fps ~ 7.9
fps ~ 7.9
fps ~ 7.9
fps ~ 7.9
fps ~ 7.9
fps ~ 7.9
fps ~ 7.9
fps ~ 7.9
fps ~ 7.9
fps ~ 7.9
fps ~ 8.0
fps ~ 8.0
fps ~ 8.0
fps ~ 8.0
fps ~ 8.0
fps ~ 8.0
fps ~ 8.0
fps ~ 8.0
fps ~ 8.0
fps ~ 8.0
fps ~ 8.0
fps ~ 8.0
fps ~ 8.0
fps ~ 8.0
fps ~ 8.0
fps ~ 8.0
fps ~ 8.0
fps ~ 8.0
fps ~ 8.0
fps ~ 8.0
fps ~ 8.0
fps ~ 8.0
fps ~ 8.0
fps ~ 8.0
fps ~ 8.0
fps ~ 8.0
fps ~ 8.0
fps ~ 8.0
fps ~ 8.0
fps ~ 8.0
fps ~ 8.0
fps ~ 8.0
fps ~ 8.0
fps ~ 8.0
fps ~ 8.0
fps ~ 8.0
fps ~ 8.0
fps ~ 8.0
fps ~ 8.0
fps ~ 8.0
fps ~ 8.0
fps ~ 8.0
fps ~ 8.0
fps ~ 8.0
fps ~ 8.0
fps ~ 8.0
fps ~ 8.0
fps ~ 8.0
fps ~ 8.0
fps ~ 8.0
fps ~ 8.0
fps ~ 8.0
fps ~ 8.0
fps ~ 8.0
fps ~ 8.0
fps ~ 8.0
fps ~ 8.0
fps ~ 8.0
fps ~ 8.0
fps ~ 8.0
fps ~ 8.0
fps ~ 8.0
fps ~ 8.0
fps ~ 8.0
fps ~ 8.0
fps ~ 8.0
fps ~ 8.0
fps ~ 8.0
fps ~ 8.0
fps ~ 8.0
fps ~ 8.0
fps ~ 8.0
fps ~ 8.0
fps ~ 8.0


## Video Test with Offset mesure

In [1]:
import time, cv2
import numpy as np
from pynq import Overlay, allocate, MMIO

ol = Overlay("/home/xilinx/jupyter_notebooks/video/design_1.bit")

WIDTH, HEIGHT, BPP = 1280, 720, 3
stride = WIDTH * BPP

fb = allocate(shape=(HEIGHT, WIDTH, BPP), dtype=np.uint8)
fb[:] = 0; fb.flush()
addr = fb.physical_address

vdma = MMIO(ol.ip_dict['axi_vdma_0']['phys_addr'], 0x100)
MM2S_DMACR, MM2S_DMASR = 0x00, 0x04
MM2S_VSIZE, MM2S_HSIZE = 0x50, 0x54
MM2S_STRIDE, MM2S_ADDR = 0x58, 0x5C

def vdma_start():
    vdma.write(MM2S_DMACR, 0x4); time.sleep(0.2)
    vdma.write(MM2S_DMACR, 0x1)
    for i in range(3):
        vdma.write(MM2S_ADDR + i*4, addr)
    vdma.write(MM2S_STRIDE, stride)
    vdma.write(MM2S_HSIZE, WIDTH * BPP)
    vdma.write(MM2S_VSIZE, HEIGHT)

In [2]:
vdma_start()
fb[:] = 0
for x in range(0, WIDTH, 128):      # graduations tous les 128 px
    fb[:, x:x+4, :] = 255
fb[:, 0:20, :] = (255, 0, 0)        # repere ROUGE = pixel 0 du framebuffer
fb.flush()
# Regarde a l'ecran : la barre ROUGE devrait etre au bord gauche.
# La distance entre le bord gauche reel et la barre rouge = OFF.
# Chaque graduation blanche = 128 px, ca t'aide a compter.

In [ ]:
OFF = 300   # <-- ta valeur mesuree

cap = cv2.VideoCapture("/dev/video0", cv2.CAP_V4L2)
time.sleep(0.5)
cap.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter_fourcc(*'MJPG'))
cap.set(cv2.CAP_PROP_FRAME_WIDTH, WIDTH)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, HEIGHT)
for _ in range(10):
    cap.read(); time.sleep(0.05)

vdma_start()

fail = 0; t0 = time.time(); n = 0
try:
    while True:
        ok, frame = cap.read()
        if not ok or frame is None:
            fail += 1
            if fail % 30 == 0: print("read fail x", fail)
            continue
        fail = 0
        if frame.shape[1] != WIDTH or frame.shape[0] != HEIGHT:
            frame = cv2.resize(frame, (WIDTH, HEIGHT))
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        fb[:] = np.roll(rgb, -OFF, axis=1)
        fb.flush()
        n += 1
        if n % 30 == 0:
            print("fps ~ {:.1f}".format(n / (time.time() - t0)))
except KeyboardInterrupt:
    pass
finally:
    cap.release()
    vdma.write(MM2S_DMACR, 0x4)
    print("stopped, DMASR = 0x{:08X}".format(vdma.read(MM2S_DMASR)))

fps ~ 7.9
fps ~ 7.9
fps ~ 7.8
fps ~ 7.8
fps ~ 7.8
fps ~ 7.8
fps ~ 7.7
fps ~ 7.7
fps ~ 7.7
fps ~ 7.8
fps ~ 7.8
fps ~ 7.8
fps ~ 7.8
